# Real-Time Voice Cloning

This is a colab demo notebook using the open source project [CorentinJ/Real-Time-Voice-Cloning](https://github.com/CorentinJ/Real-Time-Voice-Cloning)
to clone a voice.

For other deep-learning Colab notebooks, visit [tugstugi/dl-colab-notebooks](https://github.com/tugstugi/dl-colab-notebooks).


Original issue: https://github.com/tugstugi/dl-colab-notebooks/issues/18

## Setup CorentinJ/Real-Time-Voice-Cloning

In [1]:
#@title Setup CorentinJ/Real-Time-Voice-Cloning

import os
from os.path import exists, join, basename, splitext
from pathlib import Path
import sys
import importlib
import zipfile

# Step 1: Aggressive Librosa Fix - Uninstall and force-reinstall 0.7.0
print("Desinstalando qualquer versão existente do librosa...")
!pip uninstall -y librosa > /dev/null 2>&1 || true

# Ensure compatible 'deprecated' package for librosa 0.7.0
print("Desinstalando qualquer versão existente do pacote 'deprecated'...")
!pip uninstall -y deprecated > /dev/null 2>&1 || true
print("Instalando 'deprecated==1.1.0' para compatibilidade com librosa 0.7.0...")
!pip install deprecated==1.1.0

# Ensure compatible 'numba' package for librosa 0.7.0
print("Desinstalando qualquer versão existente do pacote 'numba'...")
!pip uninstall -y numba > /dev/null 2>&1 || true
print("Instalando 'numba==0.48.0' para compatibilidade com librosa 0.7.0...")
!pip install numba==0.48.0

print("Forçando instalação do librosa==0.7.0 (verbose output)...")
!pip install librosa==0.7.0 --force-reinstall

# Force reload the module if it was already imported
if 'librosa' in sys.modules:
    del sys.modules['librosa']
import librosa
print(f"librosa version (after aggressive install): {librosa.__version__}")
print(f"librosa path: {librosa.__file__}")

# Step 2: Install other non-conflicting dependencies
!pip install -q unidecode

git_repo_url = 'https://github.com/CorentinJ/Real-Time-Voice-Cloning.git'
project_name = splitext(basename(git_repo_url))[0]

if not exists(project_name):
  print(f"Cloning {project_name}...")
  !git clone -q --recursive {git_repo_url}

  # Step 3: Modify requirements.txt to ensure librosa==0.7.0
  # This is a safeguard, as the aggressive install should already cover it.
  requirements_path = Path(project_name) / "requirements.txt"
  if requirements_path.exists():
    print(f"Modifying {requirements_path} to pin librosa==0.7.0")
    with open(requirements_path, 'r') as f:
      lines = f.readlines()
    with open(requirements_path, 'w') as f:
      found_librosa = False
      for line in lines:
        if line.strip().startswith("librosa"): # Catch lines starting with librosa
          f.write("librosa==0.7.0\n")
          found_librosa = True
        else:
          f.write(line)
      if not found_librosa:
          f.write("librosa==0.7.0\n") # Add if not found
  else:
      print(f"WARNING: {requirements_path} not found. Librosa version might not be correctly pinned by requirements.")

  # Install project specific requirements (now with potentially modified librosa entry)
  !cd {project_name} && pip install -q -r requirements.txt
  !apt-get install -qq libportaudio2
  !pip install -q https://github.com/tugstugi/dl-colab-notebooks/archive/colab_utils.zip


# Ensure model directories exist
model_dir = Path(project_name) / "saved_models" / "default"
model_dir.mkdir(parents=True, exist_ok=True)

# Function to download models using wget with improved resilience
def download_model_robust(url, output_path, model_name):
    # Check if model already exists and is not 0 bytes
    if output_path.exists() and output_path.stat().st_size > 100000: # Heuristic: >100KB
        print(f"Modelo {model_name} já existe e está completo. Pulando download.")
        return

    # Clean up potentially incomplete previous downloads
    if output_path.exists():
        print(f"Removendo arquivo incompleto ou vazio: {output_path}")
        output_path.unlink()

    print(f"Baixando {model_name} de {url}...")
    # Use wget with retries and no quiet mode for better debugging
    !wget --retry-connrefused --tries=5 --show-progress --no-check-certificate -O "{str(output_path)}" "{url}"

    if not (output_path.exists() and output_path.stat().st_size > 100000): # Check again after download
        print(f"ERRO: Download de {model_name} falhou ou arquivo {output_path} está incompleto/vazio.")
    else:
        print(f"Download de {model_name} concluído com sucesso.")


# Main download calls using blue-fish fork direct links (v1.0 models)
# These are known to be compatible with CorentinJ's original project structure for inference
# NOTE: These links previously resulted in 404s. Re-testing with improved wget parameters.
print("Verificando e baixando modelos pré-treinados...")
download_model_robust(
    "https://github.com/blue-fish/Real-Time-Voice-Cloning/releases/download/v1.0/encoder.pt",
    model_dir / "encoder.pt", "encoder.pt"
)
download_model_robust(
    "https://github.com/blue-fish/Real-Time-Voice-Cloning/releases/download/v1.0/synthesizer.pt",
    model_dir / "synthesizer.pt", "synthesizer.pt"
)
download_model_robust(
    "https://github.com/blue-fish/Real-Time-Voice-Cloning/releases/download/v1.0/vocoder.pt",
    model_dir / "vocoder.pt", "vocoder.pt"
)


if project_name not in sys.path:
  sys.path.append(project_name)

# Import project modules. These will now use the correctly installed librosa version.
from synthesizer.inference import Synthesizer
from encoder import inference as encoder
from vocoder import inference as vocoder
from IPython.display import display, Audio, clear_output
import ipywidgets as widgets
import numpy as np
from dl_colab_notebooks.audio import record_audio, upload_audio

# Force reload modules to ensure they use the correct librosa version
importlib.reload(encoder)
importlib.reload(vocoder)

# Check and Load models
all_models_loaded = True
required_models = ["encoder.pt", "synthesizer.pt", "vocoder.pt"]
for model_file in required_models:
    f_path = model_dir / model_file
    if not (f_path.exists() and f_path.stat().st_size > 100000): # Check if file exists and is not too small
        print(f"ERRO: O arquivo de modelo {model_file} não foi encontrado ou está incompleto. Verifique os downloads.")
        all_models_loaded = False
        break

if all_models_loaded:
    try:
        encoder.load_model(model_dir / "encoder.pt")
        synthesizer = Synthesizer(model_dir / "synthesizer.pt")
        vocoder.load_model(model_dir / "vocoder.pt")
        print("Modelos carregados com sucesso!")
    except Exception as e:
        print(f"ERRO ao carregar os modelos: {e}")
        print("Por favor, verifique a integridade dos arquivos baixados e tente executar novamente.")
else:
    print("Não foi possível carregar os modelos devido a arquivos ausentes ou incompletos.")

Desinstalando qualquer versão existente do librosa...
Desinstalando qualquer versão existente do pacote 'deprecated'...
Instalando 'deprecated==1.1.0' para compatibilidade com librosa 0.7.0...
  Using cached Deprecated-1.1.0-py2.py3-none-any.whl.metadata (2.6 kB)
Using cached Deprecated-1.1.0-py2.py3-none-any.whl (4.7 kB)
Forçando instalação do librosa==0.7.0 (verbose output)...
  Using cached librosa-0.7.0-py3-none-any.whl
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached decorator-5.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata 

ModuleNotFoundError: No module named 'numba.decorators'

In [ ]:
import os
from pathlib import Path

# Define o caminho dos modelos
model_path = Path("Real-Time-Voice-Cloning/saved_models/default")

if model_path.exists():
    print(f"Verificando diretório: {model_path}\n")
    files = os.listdir(model_path)
    if not files:
        print("O diretório está vazio.")
    for f in files:
        f_path = model_path / f
        size_mb = f_path.stat().st_size / (1024 * 1024)
        print(f"- {f}: {size_mb:.2f} MB")
else:
    print("O diretório de modelos ainda não foi criado. Execute a célula de Setup primeiro.")

In [ ]:
#@title Record or Upload
#@markdown * Either record audio from microphone or upload audio from file (.mp3, .wav, or .ogg)

import librosa
import numpy as np
from google.colab import files

SAMPLE_RATE = 22050
record_or_upload = "Upload (.mp3, .wav, or .ogg)" #@param ["Record", "Upload (.mp3, .wav, or .ogg)"]
record_seconds =   10#@param {type:"number", min:1, max:10, step:1}

embedding = None

def _compute_embedding(audio):
  display(Audio(audio, rate=SAMPLE_RATE, autoplay=True))
  global embedding
  embedding = None
  # Preprocess and compute the speaker embedding
  processed_wav = encoder.preprocess_wav(audio, SAMPLE_RATE)
  embedding = encoder.embed_utterance(processed_wav)
  print("Embedding calculado com sucesso!")

def _record_audio(b):
  clear_output()
  print("Gravando...")
  audio = record_audio(record_seconds, sample_rate=SAMPLE_RATE)
  _compute_embedding(audio)

def _upload_audio(b):
  clear_output()
  print("Selecione seu arquivo de áudio (.ogg, .mp3, .wav):")
  uploaded = files.upload()
  for fn in uploaded.keys():
    print(f"Processando {fn}...")
    # librosa manages .ogg, .mp3, and .wav decoding
    audio, _ = librosa.load(fn, sr=SAMPLE_RATE)
    _compute_embedding(audio)

if record_or_upload == "Record":
  button = widgets.Button(description="Record Your Voice")
  button.on_click(_record_audio)
  display(button)
else:
  # Directly call upload if selected
  _upload_audio(None)

In [ ]:
#@title Synthesize a text { run: "auto" }
text = "One of the two people who tested positive for the novel coronavirus in the United Kingdom is a student at the University of York in northern England." #@param {type:"string"}

def synthesize(embed, text):
  print("Synthesizing new audio...")
  #with io.capture_output() as captured:
  specs = synthesizer.synthesize_spectrograms([text], [embed])
  generated_wav = vocoder.infer_waveform(specs[0])
  generated_wav = np.pad(generated_wav, (0, synthesizer.sample_rate), mode="constant")
  clear_output()
  display(Audio(generated_wav, rate=synthesizer.sample_rate, autoplay=True))

if embedding is None:
  print("first record a voice or upload a voice file!")
else:
  synthesize(embedding, text)